<a href="https://colab.research.google.com/github/gaallmin/Prediction-of-Agricultural-Product-prices/blob/yeji/2%EC%B0%A8%EC%98%88%EC%84%A0/SARIMA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive/')
%cd /content/drive/MyDrive/Dacon/물가 예측 2차

Mounted at /content/drive/
/content/drive/MyDrive/Dacon/물가 예측 2차


In [38]:
import pandas as pd

# Load the data
df1 = pd.read_csv('./train/train_1.csv')
df2 = pd.read_csv('./train/train_2.csv')

# Define the date parsing function
def parse_date(date):
    year = int(date[:4])
    month = int(date[4:6])
    period = date[6:]

    if period == '하순':
        day = 5
    elif period == '중순':
        day = 15
    elif period == '상순':
        day = 25
    else:
        day = 1

    return pd.Timestamp(year=year, month=month, day=day)

# Apply date parsing to both datasets
df1['datetime'] = df1['YYYYMMSOON'].apply(parse_date)
df2['datetime'] = df2['YYYYMMSOON'].apply(parse_date)

# Create separate DataFrames for each unique item
# Use '품목(품종)명' for df1 and '품목명' for df2
item_dfs1 = {item: df1[df1['품목(품종)명'] == item][['datetime', '평균가격(원)']].copy() for item in df1['품목(품종)명'].unique()}
item_dfs2 = {item: df2[df2['품목명'] == item][['datetime', '평균가격(원)']].copy() for item in df2['품목명'].unique()}

# Merge both item_dfs dictionaries based on unique item names
merged_item_dfs = {}

for item in set(item_dfs1.keys()).union(item_dfs2.keys()):
    # Get data for the current item from both df1 and df2, or an empty DataFrame if it doesn't exist in one of them
    item_df1 = item_dfs1.get(item, pd.DataFrame(columns=['datetime', '평균가격(원)']))
    item_df2 = item_dfs2.get(item, pd.DataFrame(columns=['datetime', '평균가격(원)']))

    # Concatenate data for the item and sort by datetime
    merged_item_dfs[item] = pd.concat([item_df1, item_df2]).sort_values(by='datetime').reset_index(drop=True)

<ipython-input-38-4f55fe17f559>:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged_item_dfs[item] = pd.concat([item_df1, item_df2]).sort_values(by='datetime').reset_index(drop=True)
<ipython-input-38-4f55fe17f559>:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged_item_dfs[item] = pd.concat([item_df1, item_df2]).sort_values(by='datetime').reset_index(drop=True)
<ipython-input-38-4f55fe17f559>:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is d

In [39]:
merged_item_dfs

{'대파(일반)':       datetime      평균가격(원)
 0   2018-01-05  1834.333333
 1   2018-01-15  1624.888889
 2   2018-01-25  1685.285714
 3   2018-02-05  1444.285714
 4   2018-02-15  1537.833333
 ..         ...          ...
 175 2022-11-15  1927.500000
 176 2022-11-25  1867.222222
 177 2022-12-05  2045.200000
 178 2022-12-15  1782.375000
 179 2022-12-25  1597.444444
 
 [180 rows x 2 columns],
 '무':       datetime       평균가격(원)
 0   2018-01-05  10576.111111
 1   2018-01-15   9259.888889
 2   2018-01-25   9283.571429
 3   2018-02-05  19226.000000
 4   2018-02-15  19545.666667
 ..         ...           ...
 175 2022-11-15  11933.375000
 176 2022-11-25  13996.333333
 177 2022-12-05  12959.200000
 178 2022-12-15  11222.500000
 179 2022-12-25  11416.111111
 
 [180 rows x 2 columns],
 '배':       datetime  평균가격(원)
 0   2018-01-05  28324.0
 1   2018-01-15  28290.0
 2   2018-01-25  28312.0
 3   2018-02-05  28129.0
 4   2018-02-15  28294.0
 ..         ...      ...
 175 2022-11-15  25345.0
 176 2022-11-25  2

In [34]:
from sklearn.model_selection import train_test_split

# Split each item DataFrame into train and test
train_test_data = {}
for item, data in merged_item_dfs.items():
    train, test = train_test_split(data, test_size=0.2, shuffle=False)
    train_test_data[item] = {'train': train, 'test': test}

In [35]:
import statsmodels.api as sm

models = {}

for item, data in train_test_data.items():
    # y_train의 인덱스를 datetime으로 설정하고 빈도를 추론
    y_train = data['train'].set_index('datetime')['평균가격(원)']
    y_train = y_train.resample('ME').mean()  # 'ME'를 사용해 월별 리샘플링

    # SARIMA 모델 학습
    sarima_model = sm.tsa.statespace.SARIMAX(
        y_train,
        order=(1, 1, 1),  # ARIMA 순서 (p, d, q)
        seasonal_order=(1, 1, 1, 4),  # SARIMA 계절적 순서 (P, D, Q, s)
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(disp=False)

    models[item] = sarima_model


In [36]:
# Forecasting on test set
predictions = {}
for item, model in models.items():
    test_data = train_test_data[item]['test']
    forecast = model.forecast(steps=len(test_data))
    predictions[item] = forecast

In [37]:
from sklearn.metrics import mean_absolute_error
import numpy as np

# NMAE 계산 함수 정의
def nmae_not_dict(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    nmae = mae / np.mean(np.abs(y_true))  # 정규화된 MAE 계산
    return nmae

# 테스트 데이터셋에 대한 NMAE 성능 평가
def evaluate_predictions(predictions, train_test_data):
    total_score = 0
    item_count = len(predictions)

    for item, forecast in predictions.items():
        # 실제 테스트 데이터셋 값 불러오기
        y_true = train_test_data[item]['test']['평균가격(원)'].values

        # 예측값과 실제값의 NMAE 계산
        nmae_score = nmae_not_dict(y_true, forecast)
        print(f"{item}: NMAE = {nmae_score}")

        # 총 NMAE 계산을 위해 합산
        total_score += nmae_score

    # 전체 평균 NMAE 출력
    average_nmae = total_score / item_count
    print(f"Average NMAE across all items: {average_nmae}")

# NMAE 성능 평가 함수 호출
evaluate_predictions(predictions, train_test_data)

대파(일반): NMAE = 0.5526211454751653
무: NMAE = 0.41912395492253673
배: NMAE = 1.1572761325556409
감자 수미: NMAE = 0.21763903844465485
깐마늘(국산): NMAE = 0.19071389391603605
건고추: NMAE = 0.8375453782501057
상추: NMAE = 0.30844880432521027
사과: NMAE = 0.5472993954731962
양파: NMAE = 0.6170041809594005
배추: NMAE = 0.34777492649541736
Average NMAE across all items: 0.5195446850817362
